# Geo-Explainer on Colab (free: Edge-TTS voice, Whisper timing, ESRI satellite tiles)

Run the cells top to bottom. No API keys needed.

1. **Setup** (about 3-5 minutes, once per session)
2. **Paste your script** (ask Claude for a SceneGraph JSON, see the prompt in the last cell)
3. **Preview** one frame, then **render** the MP4

Tip: `Runtime > Change runtime type > T4 GPU` lets both the word-timing step and (if the GPU check in step 4 says so) the map drawing use the GPU.

## 1. Setup

In [ ]:
%%bash
set -e
cd /content
if [ -d geo-explainer-mcp ]; then git -C geo-explainer-mcp pull -q; else git clone -q https://github.com/ramiz-moktader/geo-explainer-mcp.git; fi

# Node.js 18+ is required by the renderer
NODE_MAJOR=$( (node -v 2>/dev/null || echo v0) | sed 's/v\([0-9]*\).*/\1/')
if [ "$NODE_MAJOR" -lt 18 ]; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null
  apt-get -qq install -y nodejs >/dev/null
fi
echo "node $(node -v)"

# system libraries headless Chrome needs
apt-get -qq update >/dev/null
# installed one by one so a renamed package (e.g. libasound2 -> libasound2t64 on Ubuntu 24.04) cannot block the rest
for pkg in libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libgbm1 libxkbcommon0 libxcomposite1 \
  libxdamage1 libxfixes3 libxrandr2 libpango-1.0-0 libcairo2 fonts-liberation; do
  apt-get -qq install -y "$pkg" >/dev/null 2>&1 || apt-get -qq install -y "${pkg}t64" >/dev/null 2>&1 || echo "note: could not install $pkg"
done
apt-get -qq install -y libasound2t64 >/dev/null 2>&1 || apt-get -qq install -y libasound2 >/dev/null 2>&1 || echo "note: could not install libasound2"

pip install -q -e /content/geo-explainer-mcp
cd /content/geo-explainer-mcp/renderer && npm install --no-audit --no-fund --silent
mkdir -p node_modules && echo ok > node_modules/.geo-install-ok
echo "setup done"

In [ ]:
# Which regions, props and sound effects can a script use?
import json, subprocess
cat = json.loads(subprocess.run(["python", "-c", "import json;from geo_mcp.asset_catalog import build_catalog;print(json.dumps(build_catalog()))"],
                                capture_output=True, text=True, check=True).stdout)
for g in cat["geojson"]:
    print(g["file"], "->", ", ".join(f'{f["id"]} ({f["name"]})' for f in g["features"][:40]))
print("props:", cat["props"]); print("sfx:", cat["sfx"])

### Optional: add your own boundaries
Upload `.geojson` files (each feature needs a `name`, `ISO_A3` or `ISO_3166_2` property) and re-run the catalog cell. To keep them permanently, commit them to `assets/geojson/` in your GitHub repo instead.

In [ ]:
from google.colab import files
import shutil
for name in files.upload():
    shutil.move(name, f"/content/geo-explainer-mcp/assets/geojson/{name}")
    print("added", name)

## 2. Your script
Replace the JSON below with the SceneGraph Claude wrote for you. As it stands, this cell writes the built-in UK example.

In [ ]:
import json
from pathlib import Path
import sys; sys.path.insert(0, "/content/geo-explainer-mcp/src")
from geo_mcp.models import SceneGraph, example_scene_graph

scene_graph = example_scene_graph()   # <- replace with:  scene_graph = { ...your JSON... }

SceneGraph.model_validate(scene_graph)   # fails here with a clear message if the JSON is wrong
Path("/content/my_script.json").write_text(json.dumps(scene_graph, indent=2))
print("ok:", scene_graph["title"], "-", len(scene_graph["scenes"]), "scenes")

## 3. Preview one frame (about 1 minute)
Check that you see **satellite imagery**. A flat blue/green map means the tiles could not be reached.

In [ ]:
SCENE = 0
!cd /content && geo-explainer-render my_script.json --preview {SCENE} --out /content/renders 2>&1 | tail -3
from IPython.display import Image
Image(f"/content/renders/preview-scene{SCENE}.png", width=320)

## 4. Render the video
You will see lines like `[rendering] 45% rendered 300/900 frames` as it works. The first run also downloads headless Chrome and the Whisper model.

If you see `warning: word timings were ESTIMATED`, Whisper did not run and captions will be only roughly in sync.

In [ ]:
# GPU check: is Chrome's WebGL (what draws the map) using the GPU?
#   "WebGL: Hardware accelerated"  -> map is drawn on the GPU
#   "WebGL: Software only ..."     -> map is drawn on the CPU (the render cell will still work, just slower)
import os, subprocess
env = dict(os.environ, GEO_GL="angle-egl", GEO_CHROME_MODE="chrome-for-testing")
out = subprocess.run(["npx", "--no-install", "remotion", "gpu"], cwd="/content/geo-explainer-mcp/renderer",
                     env=env, capture_output=True, text=True)
lines = [l for l in (out.stdout + out.stderr).splitlines() if any(k in l for k in ("WebGL:", "OpenGL:", "Rasterization:", "Compositing:", "rror"))]
print("\n".join(lines) or (out.stdout + out.stderr)[-1500:])


In [ ]:
# Progress prints live.
# DRAFT = True  -> half size (540x960), much faster. Set False for the final video.
# USE_GPU = True -> draw the map on the Colab GPU (Runtime > Change runtime type > T4 GPU).
#                  If the GPU attempt fails, the cell automatically renders again on the CPU.
DRAFT = True
USE_GPU = True

import os, shutil, subprocess
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
os.environ["GEO_RENDER_SCALE"] = "0.5" if DRAFT else "1"
os.environ["GEO_RENDER_CONCURRENCY"] = str(os.cpu_count() or 2)

def render():
    proc = subprocess.Popen(["geo-explainer-render", "my_script.json", "--out", "/content/renders"], cwd="/content",
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if "memory" not in line.lower():
            print(line, end="", flush=True)
    return proc.wait()

if USE_GPU and has_gpu:
    os.environ["GEO_GL"] = "angle-egl"
    os.environ["GEO_CHROME_MODE"] = "chrome-for-testing"
    print("GPU found: rendering with GEO_GL=angle-egl")
    if render() != 0:
        print("\nGPU render failed, retrying on CPU...\n")
        os.environ.pop("GEO_GL"); os.environ.pop("GEO_CHROME_MODE")
        render()
else:
    print("No GPU runtime (or USE_GPU=False): rendering on CPU")
    os.environ.pop("GEO_GL", None); os.environ.pop("GEO_CHROME_MODE", None)
    render()


In [ ]:
import glob, os
from IPython.display import Video
from google.colab import files
mp4 = max(glob.glob("/content/renders/*.mp4"), key=os.path.getmtime)
print(mp4)
files.download(mp4)
Video(mp4, embed=True, width=300)

## Troubleshooting
- **Voice step fails (Edge-TTS)**: re-run the cell; Microsoft's free endpoint sometimes refuses cloud IPs. If it keeps failing, render on your own computer instead (see `setup/CLAUDE_SETUP.md`).
- **Draft faster**: run `%env GEO_RENDER_SCALE=0.5` before rendering for a 540x960 draft.
- **Chrome fails to launch**: copy the error; it usually names one missing system library to add to the `apt-get` line in Setup.

## Prompt to give Claude for a script
> Write a SceneGraph JSON for a 30-second vertical map explainer about **<your topic>**. Follow this schema and example exactly: <paste the output of the cell below>. Use only these region ids: <paste the catalog output>. Every `start_word` and `trigger_word` must appear word-for-word in `narration`, and scenes must follow narration order. Return only the JSON.

In [ ]:
print(json.dumps({"json_schema": SceneGraph.model_json_schema(), "example": example_scene_graph()})[:200], "...")
Path("/content/schema_for_claude.json").write_text(json.dumps({"json_schema": SceneGraph.model_json_schema(), "example": example_scene_graph()}, indent=1))
files.download("/content/schema_for_claude.json")   # attach this file to your Claude chat